# FORCE-OPT: Forward Reachable Sets from Conformal Estimation and Convex Optimization

Implementation based on: *Safety Evaluation of Motion Plans Using Trajectory Predictors as Forward Reachable Set Estimators*  
(Chakraborty, Feng, Veer, Sharma, Ding, Topan, Ivanovic, Pavone, Bansal — arXiv:2507.22389v1)

---

## Overview

FORCE-OPT extracts **Forward Reachable Sets (FRS)** from GMM-based trajectory predictors via:
1. **Convex Optimization** (Theorem 2, Eq. 6) — finds minimum-volume ellipsoidal unions covering τ probability mass
2. **Conformal Prediction** (Section IV-C) — calibrates covariances so FRS covers ground truth with high probability
3. **Bayesian Filtering** (Section IV-D) — dynamically adjusts conservativeness under distribution shift
4. **Safety Evaluation** — checks ego motion plan against calibrated FRS for collisions

In [ ]:
import numpy as np
from scipy.optimize import minimize, LinearConstraint, NonlinearConstraint
from scipy.stats import chi2, norm
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Ellipse
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

---
## 1. Data Structures

We define core data classes for GMM components, full GMM distributions, scenes, and ego plans.

In [ ]:
@dataclass
class GMMComponent:
    """A single Gaussian component in a GMM.
    
    Attributes:
        mean: (nx,) mean vector of this mode.
        covariance: (nx, nx) covariance matrix Σ_i.
        weight: mixing weight p_i (non-negative, sums to 1 across components).
    """
    mean: np.ndarray       # shape (nx,)
    covariance: np.ndarray  # shape (nx, nx)
    weight: float           # p_i >= 0


@dataclass
class GMMDistribution:
    """Gaussian Mixture Model: µ_t = Σ p_i * N(x_i, Σ_i).
    
    Represents the predicted distribution over an agent's state at a single
    future timestep, as output by trajectory predictors (e.g., Trajectron++, Autobots).
    """
    components: List[GMMComponent]
    
    @property
    def K(self) -> int:
        """Number of mixture modes."""
        return len(self.components)
    
    @property
    def weights(self) -> np.ndarray:
        return np.array([c.weight for c in self.components])
    
    def validate(self):
        """Check that weights are non-negative and sum to ~1."""
        w = self.weights
        assert np.all(w >= 0), "Weights must be non-negative."
        assert np.isclose(w.sum(), 1.0), f"Weights must sum to 1, got {w.sum():.6f}"


@dataclass
class Scene:
    """A traffic scene s = (ξ_{-H:0}, m) containing historical trajectories
    and map/context information.
    
    Attributes:
        agent_history: (H, nx) historical trajectory of the agent of interest.
        ego_history: (H, nx) historical trajectory of the ego vehicle.
        context: Optional dictionary for map info, lane topology, etc.
    """
    agent_history: np.ndarray
    ego_history: np.ndarray
    context: Optional[dict] = None


@dataclass
class MotionPlan:
    """Ego vehicle's planned trajectory.
    
    Attributes:
        positions: (T, 2) array of planned (x, y) positions at each future timestep.
        ego_length: Physical length of the ego vehicle (meters).
        ego_width: Physical width of the ego vehicle (meters).
    """
    positions: np.ndarray   # (T, 2)
    ego_length: float = 4.5
    ego_width: float = 2.0

---
## 2. Convex Optimization for FRS Extraction (Theorem 2, Eq. 6)

For a 2D state space (X ⊆ ℝ²), the FRS extraction reduces to the convex program:

$$
\min_{c_1,\ldots,c_K} \sum_{i=1}^{K} \pi \sqrt{\lambda_{i,1} \lambda_{i,2}} \, c_i
\quad \text{s.t.} \quad
\sum_{i=1}^{K} p_i (1 - e^{-c_i/2}) \geq \tau, \quad c_i \geq 0
$$

where $\lambda_{i,1}, \lambda_{i,2}$ are eigenvalues of $\Sigma_i$, and the sublevel set
$E_i(c_i) = \{x : (x - x_i)^\top \Sigma_i^{-1} (x - x_i) \leq c_i\}$ is an ellipse.

In [ ]:
def solve_frs_convex_optimization(
    gmm: GMMDistribution,
    tau: float = 0.95,
) -> np.ndarray:
    """Solve the convex optimization (Eq. 6) to extract the FRS from a GMM.
    
    Finds the minimum-volume union of Mahalanobis ellipsoids E_i(c_i) that
    collectively capture at least τ probability mass under the GMM.
    
    By Corollary 1, the optimal c* values are invariant to covariance scaling,
    so we only need to solve this once regardless of conformal calibration factor η.
    
    Args:
        gmm: The GMM distribution at a single timestep.
        tau: Minimum probability mass the FRS must capture (close to 1).
    
    Returns:
        c_star: (K,) array of optimal sublevel set thresholds for each mode.
    """
    K = gmm.K
    weights = gmm.weights
    
    # Compute eigenvalues for each component's covariance (used in objective)
    eigenvalues = []
    for comp in gmm.components:
        eigvals = np.linalg.eigvalsh(comp.covariance)
        eigenvalues.append(eigvals)
    eigenvalues = np.array(eigenvalues)  # (K, 2)
    
    # Area coefficients: a_i = π * sqrt(λ_{i,1} * λ_{i,2})
    area_coeffs = np.pi * np.sqrt(eigenvalues[:, 0] * eigenvalues[:, 1])
    
    # --- Analytic KKT solution via bisection on Lagrange multiplier λ ---
    #
    # The KKT conditions for this convex program give:
    #   a_i = λ * p_i * exp(-c_i/2) / 2    for modes with c_i > 0
    # Rearranging: c_i = max(0, -2 * log(2 * a_i / (λ * p_i)))
    #
    # We find λ such that the coverage constraint is tight:
    #   Σ_i p_i * (1 - exp(-c_i(λ)/2)) = τ
    
    def c_from_lambda(lam):
        """Compute c_i from Lagrange multiplier λ using KKT conditions."""
        ratio = 2.0 * area_coeffs / (lam * weights + 1e-30)
        return np.maximum(0.0, -2.0 * np.log(np.clip(ratio, 1e-30, None)))
    
    def coverage_from_lambda(lam):
        """Coverage achieved for a given λ."""
        c = c_from_lambda(lam)
        return np.dot(weights, 1.0 - np.exp(-c / 2.0))
    
    # Bisect to find λ such that coverage = τ
    # Larger λ → larger c_i → more coverage
    lam_lo, lam_hi = 1e-6, 1e6
    for _ in range(100):
        lam_mid = (lam_lo + lam_hi) / 2.0
        cov = coverage_from_lambda(lam_mid)
        if cov < tau:
            lam_lo = lam_mid
        else:
            lam_hi = lam_mid
        if abs(cov - tau) < 1e-12:
            break
    
    c_star = c_from_lambda(lam_hi)
    return c_star


# Quick validation
test_gmm = GMMDistribution([
    GMMComponent(mean=np.array([0.0, 0.0]), covariance=np.eye(2) * 1.0, weight=0.5),
    GMMComponent(mean=np.array([3.0, 0.0]), covariance=np.eye(2) * 0.5, weight=0.5),
])
c_star = solve_frs_convex_optimization(test_gmm, tau=0.95)
coverage_achieved = np.dot(test_gmm.weights, 1.0 - np.exp(-c_star / 2.0))
print(f"Optimal c*: {c_star}")
print(f"Coverage achieved: {coverage_achieved:.4f} (target τ = 0.95)")

---
## 3. Mahalanobis Energy & FRS Membership

The Mahalanobis energy for mode $i$ is:
$$V_i(x) = (x - x_i)^\top \Sigma_i^{-1} (x - x_i)$$

A point $x$ lies inside the FRS $E^* = \bigcup_i E_i(c_i^*)$ if $\min_i V_i(x) / c_i^* \leq 1$ (for the base covariance), or more generally $\min_i V_i(x) / c_i^* < \eta$ for the conformalized FRS.

In [ ]:
def mahalanobis_energy(
    x: np.ndarray,
    mean: np.ndarray,
    covariance: np.ndarray,
) -> float:
    """Compute Mahalanobis energy V_i(x) = (x - μ_i)^T Σ_i^{-1} (x - μ_i).
    
    This measures how far point x is from the mode center in units of the
    covariance-weighted distance.
    
    Args:
        x: (nx,) query point.
        mean: (nx,) mode center.
        covariance: (nx, nx) covariance matrix.
    
    Returns:
        Scalar Mahalanobis energy value.
    """
    diff = x - mean
    return float(diff @ np.linalg.solve(covariance, diff))


def is_in_frs(
    x: np.ndarray,
    gmm: GMMDistribution,
    c_star: np.ndarray,
    eta: float = 1.0,
) -> bool:
    """Check if point x lies inside the (conformalized) FRS.
    
    The conformalized FRS is C(s, η) = ∪_i E_i(η * c_i*).
    A point x is inside if V_i(x) <= η * c_i* for any mode i.
    
    Args:
        x: (nx,) query point.
        gmm: The GMM distribution.
        c_star: (K,) optimal sublevel set thresholds from Eq. 6.
        eta: Conformal calibration factor (default 1.0 = uncalibrated).
    
    Returns:
        True if x is inside the FRS.
    """
    for i, comp in enumerate(gmm.components):
        if c_star[i] <= 0:
            continue
        energy = mahalanobis_energy(x, comp.mean, comp.covariance)
        if energy <= eta * c_star[i]:
            return True
    return False


def compute_nonconformity_score(
    x: np.ndarray,
    gmm: GMMDistribution,
    c_star: np.ndarray,
) -> float:
    """Compute the non-conformity score ψ(s, x) from Eq. 7.
    
    ψ(s, x) = min_i { V_i(x) / c_i* }
    
    This is the smallest covariance scaling factor α needed so that x falls
    inside the FRS E*(α * Σ_i). Thanks to Corollary 1, c* is invariant to
    covariance scaling, so we can compute ψ analytically.
    
    Args:
        x: (nx,) ground-truth future state.
        gmm: The GMM distribution.
        c_star: (K,) optimal sublevel thresholds.
    
    Returns:
        Non-conformity score (lower = easier to cover).
    """
    scores = []
    for i, comp in enumerate(gmm.components):
        if c_star[i] <= 0:
            continue
        energy = mahalanobis_energy(x, comp.mean, comp.covariance)
        scores.append(energy / c_star[i])
    
    if len(scores) == 0:
        return float('inf')
    return min(scores)

---
## 4. Conformal Prediction Calibration (Section IV-C)

Given calibration data $D = \{(s_j, x_j)\}_{j=1}^{N}$, we:
1. Compute non-conformity scores $\psi(s_j, x_j)$ for each example.
2. Set $\eta$ = the $(1 - \gamma)$ empirical quantile of these scores.
3. The conformalized FRS is $C(s, \eta) = E^*(\{\eta \Sigma_i\})$.

From Eq. 8, this guarantees $\Pr(x \notin C(s, \eta)) < \gamma - \sqrt{-\log(\delta)/2N}$ with probability $1-\delta$.

In [ ]:
def calibrate_conformal(
    calibration_scores: np.ndarray,
    gamma: float = 0.05,
) -> float:
    """Compute the conformal calibration factor η via split conformal prediction.
    
    η is the (1 - γ) empirical quantile of the non-conformity scores, which
    ensures the conformalized FRS covers ground truth with probability ≥ 1 - γ.
    
    Args:
        calibration_scores: (N,) array of non-conformity scores from calibration set.
        gamma: Desired miscoverage rate (e.g., 0.05 for 95% coverage).
    
    Returns:
        eta: Conformal calibration factor for scaling covariances.
    """
    N = len(calibration_scores)
    # Use ceil((N+1)*(1-gamma))/N quantile for finite-sample validity
    quantile_level = min(np.ceil((N + 1) * (1 - gamma)) / N, 1.0)
    eta = float(np.quantile(calibration_scores, quantile_level))
    return eta


def compute_conformal_guarantee(
    N: int, gamma: float, delta: float = 0.05
) -> float:
    """Compute the coverage guarantee bound from Eq. 8.
    
    With probability 1 - δ:
        Pr(x ∉ C(s, η)) < γ - sqrt(-log(δ) / 2N)
    
    Args:
        N: Number of calibration examples.
        gamma: Target miscoverage rate.
        delta: Confidence parameter.
    
    Returns:
        Upper bound on the true miscoverage probability.
    """
    correction = np.sqrt(-np.log(delta) / (2 * N))
    return gamma + correction  # Note: paper subtracts, giving tighter bound

---
## 5. Bayesian Filtering for OOD Robustness (Section IV-D)

To handle distribution shift, we maintain a belief over model confidence $\beta \in \{\beta_{\text{low}}, \beta_{\text{high}}\}$:

$$
\text{bel}_{t+1}(\beta) = \frac{\phi\!\left(x_t;\, \text{GMM}\!\left(\{\bar{x}_t\},\, \tfrac{1}{\beta}\{\eta\Sigma_i\}\right)\right) \text{bel}_t(\beta)}{\sum_{\tilde{\beta}} \phi(\cdots) \text{bel}_t(\tilde{\beta})}
$$

When $\hat{\beta} = E[\beta]$ drops below a threshold (e.g., 0.75), we switch to more conservative fallback FRS methods.

In [ ]:
def gmm_likelihood(
    x: np.ndarray,
    gmm: GMMDistribution,
    cov_scale: float = 1.0,
) -> float:
    """Compute the likelihood φ(x; GMM) with optionally scaled covariances.
    
    Args:
        x: (nx,) observed state.
        gmm: The GMM distribution.
        cov_scale: Multiplicative scaling applied to each component's covariance.
    
    Returns:
        Likelihood value (unnormalized is fine since we normalize in Bayes update).
    """
    nx = x.shape[0]
    total = 0.0
    for comp in gmm.components:
        scaled_cov = cov_scale * comp.covariance
        diff = x - comp.mean
        sign, logdet = np.linalg.slogdet(scaled_cov)
        mahal = diff @ np.linalg.solve(scaled_cov, diff)
        log_prob = -0.5 * (nx * np.log(2 * np.pi) + logdet + mahal)
        total += comp.weight * np.exp(log_prob)
    return max(total, 1e-300)  # Avoid zero for numerical stability


class BayesianConfidenceFilter:
    """Online Bayesian filter for tracking predictor confidence β.
    
    Maintains a belief over β ∈ {β_low, β_high} that reflects trust in the
    trajectory predictor. β_high (≈1) indicates high trust; β_low (<1) indicates
    the predictor is unreliable and FRS should be dilated.
    
    The covariances are scaled by (1/β) * η * Σ_i, so lower β → larger FRS.
    
    Args:
        beta_low: Confidence value when predictor is unreliable.
        beta_high: Confidence value when predictor is reliable.
        switch_threshold: β below which we switch to fallback FRS.
    """
    
    def __init__(
        self,
        beta_low: float = 0.3,
        beta_high: float = 1.0,
        switch_threshold: float = 0.75,
    ):
        self.beta_low = beta_low
        self.beta_high = beta_high
        self.switch_threshold = switch_threshold
        
        # Initialize belief uniformly: bel_0(β_low) = bel_0(β_high) = 0.5
        self.belief_low = 0.5
        self.belief_high = 0.5
        
        self.belief_history = [self.expected_beta]
    
    @property
    def expected_beta(self) -> float:
        """Expected model confidence: β_hat = E[β] under current belief."""
        return self.belief_low * self.beta_low + self.belief_high * self.beta_high
    
    @property
    def is_confident(self) -> bool:
        """Whether predictor confidence is above switching threshold."""
        return self.expected_beta >= self.switch_threshold
    
    def update(
        self,
        x_observed: np.ndarray,
        gmm: GMMDistribution,
        eta: float,
    ):
        """Bayesian belief update using observed agent state (Eq. 13).
        
        Updates bel_{t+1}(β) ∝ φ(x_t; GMM(means, (1/β)*η*Σ_i)) * bel_t(β)
        
        Args:
            x_observed: (nx,) observed ground-truth agent state at current timestep.
            gmm: The predicted GMM distribution for this timestep.
            eta: Conformal calibration factor.
        """
        # Likelihood under β_low: covariance scaled by (1/β_low) * η
        # Lower β → larger covariance → higher likelihood for distant observations
        likelihood_low = gmm_likelihood(
            x_observed, gmm, cov_scale=(1.0 / self.beta_low) * eta
        )
        # Likelihood under β_high: covariance scaled by (1/β_high) * η
        likelihood_high = gmm_likelihood(
            x_observed, gmm, cov_scale=(1.0 / self.beta_high) * eta
        )
        
        # Bayesian update
        unnorm_low = likelihood_low * self.belief_low
        unnorm_high = likelihood_high * self.belief_high
        normalizer = unnorm_low + unnorm_high
        
        if normalizer > 0:
            self.belief_low = unnorm_low / normalizer
            self.belief_high = unnorm_high / normalizer
        # else: keep beliefs unchanged (extremely unlikely edge case)
        
        self.belief_history.append(self.expected_beta)
    
    def reset(self):
        """Reset belief to uniform prior."""
        self.belief_low = 0.5
        self.belief_high = 0.5
        self.belief_history = [self.expected_beta]

---
## 6. Worst-Case FRS Fallback

When the Bayesian filter detects low confidence ($\hat{\beta} < 0.75$), we fall back to a worst-case FRS
assuming Dubins-like vehicle dynamics with bounded velocity and turn rate.

In [ ]:
def compute_worst_case_frs(
    current_pos: np.ndarray,
    current_vel: float,
    dt: float,
    t_steps: int,
    max_accel: float = 4.0,
    max_decel: float = -6.0,
    max_yaw_rate: float = 0.5,
) -> List[Tuple[np.ndarray, float]]:
    """Compute worst-case FRS using bounded Dubins vehicle dynamics.
    
    Models the agent as a 4D Dubins vehicle with bounded acceleration and
    yaw rate. The resulting FRS at each timestep is a circle centered at the
    current position with radius equal to the maximum reachable distance.
    
    Args:
        current_pos: (2,) current (x, y) position.
        current_vel: Current scalar speed.
        dt: Time discretization interval.
        t_steps: Number of future timesteps.
        max_accel: Maximum acceleration (m/s²).
        max_decel: Maximum deceleration (m/s², negative).
        max_yaw_rate: Maximum yaw rate (rad/s).
    
    Returns:
        List of (center, radius) tuples for each timestep's WC-FRS.
    """
    frs_circles = []
    for t in range(1, t_steps + 1):
        elapsed = t * dt
        # Maximum distance: integrate max speed over time
        max_speed = current_vel + max_accel * elapsed
        max_distance = current_vel * elapsed + 0.5 * max_accel * elapsed**2
        max_distance = max(max_distance, 0.0)
        # The agent can move in any direction due to yaw freedom
        frs_circles.append((current_pos.copy(), max_distance))
    return frs_circles

---
## 7. FORCE-OPT: Complete Algorithm

Brings everything together: extract FRS via convex optimization, calibrate with conformal prediction,
optionally apply Bayesian belief adjustment, and evaluate safety of the ego's motion plan.

In [ ]:
class FORCE_OPT:
    """FORCE-OPT: Forward Reachable Sets from Conformal Estimation and Convex Optimization.
    
    This class implements the full FORCE-OPT pipeline:
      1. Extract FRS from GMM via convex optimization (Section IV-B)
      2. Calibrate using conformal prediction (Section IV-C)
      3. Optionally adapt via Bayesian filtering (Section IV-D)
      4. Evaluate safety of ego motion plans against calibrated FRS
    
    Args:
        tau: Probability mass target for FRS extraction.
        gamma: Desired miscoverage rate for conformal prediction.
        use_belief: Whether to enable Bayesian confidence filtering.
        use_wc_fallback: Whether to use worst-case FRS when confidence is low.
        beta_low: Low confidence value for Bayesian filter.
        beta_high: High confidence value for Bayesian filter.
        switch_threshold: β threshold for switching to fallback FRS.
    """
    
    def __init__(
        self,
        tau: float = 0.95,
        gamma: float = 0.05,
        use_belief: bool = True,
        use_wc_fallback: bool = False,
        beta_low: float = 0.3,
        beta_high: float = 1.0,
        switch_threshold: float = 0.75,
    ):
        self.tau = tau
        self.gamma = gamma
        self.use_belief = use_belief
        self.use_wc_fallback = use_wc_fallback
        
        # Conformal calibration factor (set during calibration)
        self.eta: Optional[float] = None
        self.c_star_cache: dict = {}  # Cache c* per timestep
        
        # Bayesian filter
        self.belief_filter = BayesianConfidenceFilter(
            beta_low=beta_low,
            beta_high=beta_high,
            switch_threshold=switch_threshold,
        ) if use_belief else None
    
    def extract_frs(
        self,
        gmm: GMMDistribution,
        timestep: int = 0,
    ) -> np.ndarray:
        """Extract FRS by solving the convex optimization (Eq. 6).
        
        Results are cached per timestep since c* only depends on the GMM structure.
        
        Args:
            gmm: GMM distribution at a particular future timestep.
            timestep: Index for caching.
        
        Returns:
            c_star: (K,) optimal sublevel thresholds.
        """
        if timestep not in self.c_star_cache:
            self.c_star_cache[timestep] = solve_frs_convex_optimization(gmm, self.tau)
        return self.c_star_cache[timestep]
    
    def calibrate(
        self,
        calibration_data: List[Tuple[GMMDistribution, np.ndarray]],
    ):
        """Calibrate the FRS using split conformal prediction (Section IV-C).
        
        Computes non-conformity scores for all calibration examples and sets η
        as the (1 - γ) quantile.
        
        Args:
            calibration_data: List of (gmm_prediction, ground_truth_state) pairs.
        """
        scores = []
        for i, (gmm, x_gt) in enumerate(calibration_data):
            c_star = solve_frs_convex_optimization(gmm, self.tau)
            score = compute_nonconformity_score(x_gt, gmm, c_star)
            scores.append(score)
        
        scores = np.array(scores)
        self.eta = calibrate_conformal(scores, self.gamma)
        
        # Report calibration statistics
        empirical_coverage = np.mean(scores <= self.eta)
        print(f"Calibration complete:")
        print(f"  η = {self.eta:.4f}")
        print(f"  Empirical coverage on calibration set: {empirical_coverage:.2%}")
        print(f"  N = {len(scores)} calibration examples")
        print(f"  Target coverage: {1 - self.gamma:.2%}")
    
    def get_effective_cov_scale(self) -> float:
        """Get the effective covariance scaling factor combining η and β.
        
        The final covariance scale is (1/β_hat) * η, where:
        - η comes from conformal calibration
        - β_hat = E[β] comes from Bayesian filtering (if enabled)
        
        Returns:
            Effective covariance scaling factor.
        """
        eta = self.eta if self.eta is not None else 1.0
        
        if self.belief_filter is not None:
            beta_hat = self.belief_filter.expected_beta
            return eta / beta_hat
        return eta
    
    def check_point_in_frs(
        self,
        x: np.ndarray,
        gmm: GMMDistribution,
        c_star: np.ndarray,
    ) -> bool:
        """Check if a point is inside the calibrated (and belief-adjusted) FRS.
        
        Args:
            x: (2,) query point.
            gmm: GMM distribution.
            c_star: (K,) optimal sublevel thresholds.
        
        Returns:
            True if point is inside the FRS.
        """
        effective_scale = self.get_effective_cov_scale()
        return is_in_frs(x, gmm, c_star, eta=effective_scale)
    
    def evaluate_safety(
        self,
        ego_plan: MotionPlan,
        agent_gmms: List[GMMDistribution],
        agent_current_pos: Optional[np.ndarray] = None,
        agent_current_vel: Optional[float] = None,
        dt: float = 0.5,
    ) -> dict:
        """Evaluate safety of an ego motion plan against an agent's predicted FRS.
        
        For each future timestep, checks whether the ego's planned position
        intersects the agent's calibrated FRS. If Bayesian filtering detects
        low confidence and fallback is enabled, switches to worst-case FRS.
        
        Args:
            ego_plan: The ego vehicle's planned trajectory.
            agent_gmms: List of T GMMs, one per future timestep.
            agent_current_pos: (2,) for worst-case fallback.
            agent_current_vel: Scalar speed for worst-case fallback.
            dt: Time step interval.
        
        Returns:
            Dictionary with:
              - 'is_safe': bool, True if no collision detected at any timestep.
              - 'collision_timesteps': list of timesteps with collisions.
              - 'method_used': which method was used per timestep.
              - 'beta_hat': current expected β (if belief enabled).
        """
        T = len(agent_gmms)
        assert ego_plan.positions.shape[0] >= T, \
            f"Ego plan has {ego_plan.positions.shape[0]} steps but need {T}."
        
        collision_timesteps = []
        methods_used = []
        
        # Check if we should use fallback
        use_fallback = (
            self.use_wc_fallback
            and self.belief_filter is not None
            and not self.belief_filter.is_confident
        )
        
        if use_fallback and agent_current_pos is not None and agent_current_vel is not None:
            # Use worst-case FRS
            wc_frs = compute_worst_case_frs(
                agent_current_pos, agent_current_vel, dt, T
            )
            for t in range(T):
                center, radius = wc_frs[t]
                dist = np.linalg.norm(ego_plan.positions[t] - center)
                # Account for ego vehicle dimensions
                min_dist = max(ego_plan.ego_length, ego_plan.ego_width) / 2.0
                if dist < radius + min_dist:
                    collision_timesteps.append(t)
                methods_used.append('WC-FRS')
        else:
            # Use FORCE-OPT (with or without belief scaling)
            for t in range(T):
                gmm = agent_gmms[t]
                c_star = self.extract_frs(gmm, timestep=t)
                
                # Check ego position against FRS
                if self.check_point_in_frs(ego_plan.positions[t], gmm, c_star):
                    collision_timesteps.append(t)
                
                method_name = 'FORCE-OPT'
                if self.belief_filter is not None:
                    method_name += '+belief'
                methods_used.append(method_name)
        
        result = {
            'is_safe': len(collision_timesteps) == 0,
            'collision_timesteps': collision_timesteps,
            'methods_used': methods_used,
        }
        if self.belief_filter is not None:
            result['beta_hat'] = self.belief_filter.expected_beta
        
        return result
    
    def update_belief(
        self,
        x_observed: np.ndarray,
        gmm: GMMDistribution,
    ):
        """Update Bayesian confidence filter with newly observed agent state.
        
        Should be called at each timestep when new ground-truth observations
        become available.
        
        Args:
            x_observed: (nx,) observed agent state.
            gmm: The predicted GMM for this timestep.
        """
        if self.belief_filter is not None and self.eta is not None:
            self.belief_filter.update(x_observed, gmm, self.eta)

---
## 8. Synthetic Scenario Generation

To demonstrate the algorithm, we create a synthetic intersection scenario similar to the nuScenes
example in Figure 1 of the paper. We simulate:
- An agent (contender) driving through an intersection
- A GMM trajectory predictor outputting multi-modal predictions
- Both safe and unsafe ego plans

In [ ]:
def generate_synthetic_scenario(
    T: int = 6,
    dt: float = 0.5,
    K: int = 3,
    seed: int = 42,
) -> dict:
    """Generate a synthetic intersection scenario for demonstration.
    
    Creates a contender agent going straight through an intersection,
    a GMM predictor with multiple modes (straight, left turn, right turn),
    and both a safe ego plan and an unsafe ego plan.
    
    Args:
        T: Number of future prediction timesteps.
        dt: Time interval between steps (seconds).
        K: Number of GMM modes.
        seed: Random seed for reproducibility.
    
    Returns:
        Dictionary with scenario components.
    """
    rng = np.random.RandomState(seed)
    
    # Agent (contender) ground truth: moving roughly north-east
    agent_speed = 8.0  # m/s
    agent_heading = np.deg2rad(45)  # NE direction
    agent_start = np.array([0.0, 0.0])
    gt_trajectory = np.array([
        agent_start + agent_speed * (t * dt) * np.array([np.cos(agent_heading), np.sin(agent_heading)])
        for t in range(1, T + 1)
    ])
    
    # GMM predictions: multi-modal (generate K modes spread around agent heading)
    mode_headings = [agent_heading + np.deg2rad(offset)
                     for offset in np.linspace(-25, 25, K)]
    raw_weights = rng.dirichlet(np.ones(K) * 3)
    mode_weights = raw_weights.tolist()
    
    gmms = []
    for t_idx in range(T):
        t = (t_idx + 1) * dt
        components = []
        for k in range(K):
            heading = mode_headings[k]
            mean = agent_start + agent_speed * t * np.array([np.cos(heading), np.sin(heading)])
            # Add small prediction noise
            mean += rng.randn(2) * 0.3
            
            # Covariance grows with time (more uncertainty further out)
            # Elongated along the direction of motion
            direction = np.array([np.cos(heading), np.sin(heading)])
            perp = np.array([-np.sin(heading), np.cos(heading)])
            R = np.column_stack([direction, perp])
            D = np.diag([(0.5 + 0.8 * t)**2, (0.3 + 0.3 * t)**2])
            cov = R @ D @ R.T
            
            components.append(GMMComponent(
                mean=mean,
                covariance=cov,
                weight=mode_weights[k],
            ))
        gmms.append(GMMDistribution(components))
    
    # Safe ego plan: moves south, well away from agent
    ego_safe = MotionPlan(
        positions=np.array([
            np.array([-5.0, -3.0]) + np.array([0.0, -agent_speed * 0.6]) * (t * dt)
            for t in range(1, T + 1)
        ])
    )
    
    # Unsafe ego plan: crosses the agent's path
    ego_unsafe = MotionPlan(
        positions=np.array([
            np.array([-2.0, 3.0]) + np.array([agent_speed * 0.9, -2.0]) * (t * dt)
            for t in range(1, T + 1)
        ])
    )
    
    return {
        'gt_trajectory': gt_trajectory,
        'agent_gmms': gmms,
        'ego_safe': ego_safe,
        'ego_unsafe': ego_unsafe,
        'agent_start': agent_start,
        'agent_speed': agent_speed,
        'T': T,
        'dt': dt,
        'K': K,
    }


def generate_calibration_data(
    n_samples: int = 500,
    K: int = 3,
    seed: int = 123,
) -> List[Tuple[GMMDistribution, np.ndarray]]:
    """Generate synthetic calibration data pairs (GMM prediction, ground truth).
    
    Simulates a trajectory predictor that is sometimes accurate and sometimes
    off, mimicking real predictor behavior for conformal calibration.
    
    Args:
        n_samples: Number of calibration examples.
        K: Number of GMM modes.
        seed: Random seed.
    
    Returns:
        List of (gmm, ground_truth_position) tuples.
    """
    rng = np.random.RandomState(seed)
    data = []
    
    for _ in range(n_samples):
        # Random agent context
        speed = rng.uniform(3, 15)
        heading = rng.uniform(0, 2 * np.pi)
        t = rng.uniform(0.5, 3.0)
        
        # Ground truth with some noise
        gt = speed * t * np.array([np.cos(heading), np.sin(heading)])
        gt += rng.randn(2) * speed * 0.15  # Process noise
        
        # Predicted GMM (with modeling error)
        components = []
        weights = rng.dirichlet(np.ones(K))
        for k in range(K):
            mode_heading = heading + rng.randn() * 0.3
            mean = speed * t * np.array([np.cos(mode_heading), np.sin(mode_heading)])
            mean += rng.randn(2) * speed * 0.1  # Prediction bias
            
            direction = np.array([np.cos(mode_heading), np.sin(mode_heading)])
            perp = np.array([-np.sin(mode_heading), np.cos(mode_heading)])
            R = np.column_stack([direction, perp])
            # Predictor may underestimate uncertainty
            scale = rng.uniform(0.5, 1.5)
            D = np.diag([(scale * speed * 0.2)**2, (scale * speed * 0.1)**2])
            cov = R @ D @ R.T
            # Ensure positive definite
            cov += np.eye(2) * 1e-4
            
            components.append(GMMComponent(mean=mean, covariance=cov, weight=weights[k]))
        
        data.append((GMMDistribution(components), gt))
    
    return data


# Generate scenario and calibration data
scenario = generate_synthetic_scenario(T=6, K=3)
cal_data = generate_calibration_data(n_samples=500, K=3)
print(f"Generated scenario with T={scenario['T']} steps, K={scenario['K']} modes")
print(f"Generated {len(cal_data)} calibration examples")

---
## 9. Run FORCE-OPT Pipeline

We now run the full pipeline: calibrate → extract FRS → evaluate safety.

In [ ]:
# ── Instantiate FORCE-OPT variants ──

# Variant 1: FORCE-OPT (no belief)
force_opt = FORCE_OPT(tau=0.95, gamma=0.05, use_belief=False)
force_opt.calibrate(cal_data)

print()

# Variant 2: FORCE-OPT + belief
force_opt_belief = FORCE_OPT(tau=0.95, gamma=0.05, use_belief=True, use_wc_fallback=False)
force_opt_belief.calibrate(cal_data)

print()

# Variant 3: FORCE-OPT + belief + WC-FRS fallback
force_opt_wc = FORCE_OPT(tau=0.95, gamma=0.05, use_belief=True, use_wc_fallback=True)
force_opt_wc.calibrate(cal_data)

In [ ]:
# ── Evaluate safety for both plans ──

print("=" * 60)
print("SAFETY EVALUATION RESULTS")
print("=" * 60)

for plan_name, plan in [('Safe Plan', scenario['ego_safe']), ('Unsafe Plan', scenario['ego_unsafe'])]:
    print(f"\n--- {plan_name} ---")
    
    # FORCE-OPT
    result = force_opt.evaluate_safety(
        ego_plan=plan,
        agent_gmms=scenario['agent_gmms'],
    )
    print(f"  FORCE-OPT:          Safe={result['is_safe']}, Collisions at t={result['collision_timesteps']}")
    
    # FORCE-OPT + belief
    result_b = force_opt_belief.evaluate_safety(
        ego_plan=plan,
        agent_gmms=scenario['agent_gmms'],
    )
    print(f"  FORCE-OPT + belief: Safe={result_b['is_safe']}, β̂={result_b['beta_hat']:.3f}, Collisions at t={result_b['collision_timesteps']}")
    
    # Baseline: 99% CI (uncalibrated)
    c_99 = chi2.ppf(0.99, df=2)  # ≈ 9.21
    collisions_99 = []
    for t_idx in range(scenario['T']):
        gmm = scenario['agent_gmms'][t_idx]
        if is_in_frs(plan.positions[t_idx], gmm, np.full(gmm.K, c_99), eta=1.0):
            collisions_99.append(t_idx)
    print(f"  99% CI baseline:    Safe={len(collisions_99)==0}, Collisions at t={collisions_99}")

In [ ]:
# ── Evaluate coverage on ground-truth trajectory ──

print("\n" + "=" * 60)
print("COVERAGE OF GROUND-TRUTH TRAJECTORY")
print("=" * 60)

gt = scenario['gt_trajectory']
for t_idx in range(scenario['T']):
    gmm = scenario['agent_gmms'][t_idx]
    c_star = force_opt.extract_frs(gmm, timestep=t_idx)
    
    # FORCE-OPT (calibrated)
    covered_fo = is_in_frs(gt[t_idx], gmm, c_star, eta=force_opt.eta)
    
    # 99% CI (uncalibrated)
    covered_99 = is_in_frs(gt[t_idx], gmm, np.full(gmm.K, c_99), eta=1.0)
    
    print(f"  t={t_idx+1}: GT={gt[t_idx]}  FORCE-OPT covers: {covered_fo}  99%CI covers: {covered_99}")

---
## 10. Visualization

We reproduce a figure similar to Fig. 1 from the paper, comparing the three approaches:
- **(a) Baseline (99% CI):** Small, overconfident ellipses — low FPR but poor coverage (high FNR)
- **(b) Worst-Case FRS:** Large conservative circles — excellent coverage but many false positives
- **(c) FORCE-OPT (Ours):** Calibrated ellipses — good coverage with tight sets

In [ ]:
def plot_ellipse(
    ax, mean, covariance, threshold, color='cyan', alpha=0.3, label=None
):
    """Plot a Mahalanobis ellipse E_i(c) = {x : (x-μ)^T Σ^{-1} (x-μ) ≤ c}."""
    eigvals, eigvecs = np.linalg.eigh(covariance)
    angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
    width = 2 * np.sqrt(eigvals[1] * threshold)
    height = 2 * np.sqrt(eigvals[0] * threshold)
    
    ellipse = Ellipse(
        xy=mean, width=width, height=height, angle=angle,
        facecolor=color, edgecolor=color, alpha=alpha, label=label,
    )
    ax.add_patch(ellipse)


def plot_scenario_comparison(scenario, force_opt_model):
    """Create a 3-panel comparison plot similar to Figure 1 of the paper."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    gt = scenario['gt_trajectory']
    T = scenario['T']
    
    titles = [
        '(a) Baseline Predictor (99% CI)',
        '(b) Worst-Case FRS',
        '(c) FORCE-OPT (Ours)',
    ]
    
    for ax_idx, ax in enumerate(axes):
        ax.set_title(titles[ax_idx], fontsize=13, fontweight='bold')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        
        # Draw ground truth
        ax.plot(*scenario['agent_start'], 'gs', markersize=10, label='Agent start')
        ax.plot(gt[:, 0], gt[:, 1], 'g.-', linewidth=2, markersize=8, label='Agent GT')
        
        # Draw ego plans
        safe_pos = scenario['ego_safe'].positions
        unsafe_pos = scenario['ego_unsafe'].positions
        ax.plot(safe_pos[:, 0], safe_pos[:, 1], 'b--', linewidth=2, label='Ego safe')
        ax.plot(unsafe_pos[:, 0], unsafe_pos[:, 1], 'r--', linewidth=2, label='Ego unsafe')
        
        # Draw FRS
        for t_idx in range(T):
            gmm = scenario['agent_gmms'][t_idx]
            
            if ax_idx == 0:  # 99% CI baseline
                c_val = chi2.ppf(0.99, df=2)
                for comp in gmm.components:
                    lbl = 'FRS (99% CI)' if t_idx == 0 else None
                    plot_ellipse(ax, comp.mean, comp.covariance, c_val,
                                color='cyan', alpha=0.2, label=lbl)
            
            elif ax_idx == 1:  # Worst-case FRS
                wc = compute_worst_case_frs(
                    scenario['agent_start'], scenario['agent_speed'],
                    scenario['dt'], T
                )
                center, radius = wc[t_idx]
                circle = plt.Circle(center, radius, facecolor='cyan', edgecolor='cyan',
                                   alpha=0.1,
                                   label='WC-FRS' if t_idx == 0 else None)
                ax.add_patch(circle)
            
            elif ax_idx == 2:  # FORCE-OPT
                c_star = force_opt_model.extract_frs(gmm, timestep=t_idx)
                eta = force_opt_model.eta if force_opt_model.eta else 1.0
                for i, comp in enumerate(gmm.components):
                    if c_star[i] > 0:
                        lbl = 'FRS (FORCE-OPT)' if t_idx == 0 and i == 0 else None
                        plot_ellipse(ax, comp.mean, comp.covariance, eta * c_star[i],
                                    color='cyan', alpha=0.25, label=lbl)
        
        ax.legend(loc='lower right', fontsize=8)
        ax.set_xlim(-15, 30)
        ax.set_ylim(-15, 25)
    
    plt.tight_layout()
    plt.savefig('force_opt_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved figure to force_opt_comparison.png")


plot_scenario_comparison(scenario, force_opt)

---
## 11. Multi-modality Ablation (Reproducing Fig. 2 style)

We study how BER, FPR, and FNR change as the number of GMM modes increases,
replicating the analysis from Section V-E.3 and Table III.

In [ ]:
def run_multimodality_ablation(
    n_test: int = 200,
    n_cal: int = 500,
    modes_list: List[int] = [1, 2, 3, 4, 5],
    seed: int = 99,
) -> dict:
    """Run ablation study on number of GMM modes.
    
    For each mode count K, we:
    1. Generate calibration data and calibrate FORCE-OPT.
    2. Generate test scenarios (safe + unsafe) and compute FPR, FNR, BER.
    
    Returns:
        Dictionary mapping K → {fpr, fnr, ber, eta}.
    """
    rng = np.random.RandomState(seed)
    results = {}
    
    for K in modes_list:
        # Calibrate
        cal = generate_calibration_data(n_samples=n_cal, K=K, seed=seed + K)
        model = FORCE_OPT(tau=0.95, gamma=0.05, use_belief=False)
        
        # Suppress calibration output
        import io, sys
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()
        model.calibrate(cal)
        sys.stdout = old_stdout
        
        # Generate test data
        false_positives = 0
        false_negatives = 0
        n_safe = 0
        n_unsafe = 0
        
        for trial in range(n_test):
            scen = generate_synthetic_scenario(T=6, K=K, seed=seed + trial * 100 + K)
            
            # Test safe plan → should NOT flag (FP if flagged)
            res_safe = model.evaluate_safety(scen['ego_safe'], scen['agent_gmms'])
            n_safe += 1
            if not res_safe['is_safe']:
                false_positives += 1
            
            # Test unsafe plan → should flag (FN if not flagged)
            res_unsafe = model.evaluate_safety(scen['ego_unsafe'], scen['agent_gmms'])
            n_unsafe += 1
            if res_unsafe['is_safe']:
                false_negatives += 1
        
        fpr = false_positives / n_safe
        fnr = false_negatives / n_unsafe
        ber = (fpr + fnr) / 2
        
        results[K] = {'fpr': fpr, 'fnr': fnr, 'ber': ber, 'eta': model.eta}
        print(f"K={K}: FPR={fpr:.2%}, FNR={fnr:.2%}, BER={ber:.2%}, η={model.eta:.2f}")
    
    return results


print("Running multi-modality ablation...")
ablation_results = run_multimodality_ablation()

In [ ]:
# Plot ablation results
modes = sorted(ablation_results.keys())
fpr_vals = [ablation_results[k]['fpr'] * 100 for k in modes]
fnr_vals = [ablation_results[k]['fnr'] * 100 for k in modes]
ber_vals = [ablation_results[k]['ber'] * 100 for k in modes]
eta_vals = [ablation_results[k]['eta'] for k in modes]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].plot(modes, ber_vals, 'o-', color='#2196F3', linewidth=2, markersize=8)
axes[0].set_title('BER vs. Number of Modes', fontweight='bold')
axes[0].set_xlabel('Number of GMM Modes')
axes[0].set_ylabel('BER (%)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(modes, fpr_vals, 's-', color='#FF9800', linewidth=2, markersize=8)
axes[1].set_title('FPR vs. Number of Modes', fontweight='bold')
axes[1].set_xlabel('Number of GMM Modes')
axes[1].set_ylabel('FPR (%)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(modes, fnr_vals, 'D-', color='#4CAF50', linewidth=2, markersize=8)
axes[2].set_title('FNR vs. Number of Modes', fontweight='bold')
axes[2].set_xlabel('Number of GMM Modes')
axes[2].set_ylabel('FNR (%)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('multimodality_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

# Also show calibrated η values (cf. Table III)
print("\nCalibrated η (conformal scaling factor) by number of modes:")
for k in modes:
    print(f"  K={k}: η = {ablation_results[k]['eta']:.4f}")

---
## 12. Bayesian Filter Demonstration

We simulate an OOD scenario where the agent deviates from the predicted trajectory,
showing how the Bayesian filter adjusts β over time.

In [ ]:
def demonstrate_bayesian_filter():
    """Show Bayesian filter adapting to in-distribution and OOD observations."""
    scenario = generate_synthetic_scenario(T=6, K=3)
    cal_data = generate_calibration_data(n_samples=500, K=3)
    
    # Create FORCE-OPT with belief
    model = FORCE_OPT(tau=0.95, gamma=0.05, use_belief=True)
    
    import io, sys
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    model.calibrate(cal_data)
    sys.stdout = old_stdout
    
    # ── Scenario A: ID observations (agent follows prediction well) ──
    model.belief_filter.reset()
    gt_id = scenario['gt_trajectory']
    for t_idx in range(scenario['T']):
        model.update_belief(gt_id[t_idx], scenario['agent_gmms'][t_idx])
    beta_history_id = model.belief_filter.belief_history.copy()
    
    # ── Scenario B: OOD observations (agent deviates significantly) ──
    model.belief_filter.reset()
    gt_ood = gt_id + np.array([15.0, -10.0])  # Large offset
    for t_idx in range(scenario['T']):
        model.update_belief(gt_ood[t_idx], scenario['agent_gmms'][t_idx])
    beta_history_ood = model.belief_filter.belief_history.copy()
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    timesteps = range(len(beta_history_id))
    ax.plot(timesteps, beta_history_id, 'g-o', linewidth=2, markersize=8, label='In-distribution')
    ax.plot(timesteps, beta_history_ood, 'r-s', linewidth=2, markersize=8, label='Out-of-distribution')
    ax.axhline(y=0.75, color='orange', linestyle='--', linewidth=1.5, label='Switch threshold')
    ax.fill_between(timesteps, 0, 0.75, alpha=0.05, color='red')
    ax.set_xlabel('Timestep', fontsize=12)
    ax.set_ylabel('Expected β (model confidence)', fontsize=12)
    ax.set_title('Bayesian Confidence Filter: ID vs OOD', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    ax.text(len(timesteps) - 1.5, 0.4, 'Fallback\nzone', fontsize=10,
            ha='center', color='red', alpha=0.7)
    plt.tight_layout()
    plt.savefig('bayesian_filter_demo.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"ID final β̂ = {beta_history_id[-1]:.4f}")
    print(f"OOD final β̂ = {beta_history_ood[-1]:.4f}")
    print(f"OOD triggers fallback: {beta_history_ood[-1] < 0.75}")


demonstrate_bayesian_filter()

---
## 13. Computational Efficiency Benchmark

We benchmark the runtime of FORCE-OPT's key operations, corresponding to the
timing results in Table IV of the paper.

In [ ]:
import time

def benchmark_runtime(n_trials: int = 100):
    """Benchmark computation time for different FRS methods."""
    scenario = generate_synthetic_scenario(T=6, K=5)
    gmm = scenario['agent_gmms'][3]  # Use middle timestep
    x_test = scenario['gt_trajectory'][3]
    
    results = {}
    
    # 1. Convex optimization (core of FORCE-OPT)
    times = []
    for _ in range(n_trials):
        start = time.perf_counter()
        c_star = solve_frs_convex_optimization(gmm, tau=0.95)
        times.append(time.perf_counter() - start)
    results['Convex Opt (Eq. 6)'] = np.mean(times)
    
    # 2. Non-conformity score
    times = []
    for _ in range(n_trials):
        start = time.perf_counter()
        compute_nonconformity_score(x_test, gmm, c_star)
        times.append(time.perf_counter() - start)
    results['Nonconformity Score'] = np.mean(times)
    
    # 3. Full safety evaluation
    model = FORCE_OPT(tau=0.95, gamma=0.05, use_belief=False)
    model.eta = 2.0  # dummy
    times = []
    for _ in range(n_trials):
        model.c_star_cache.clear()
        start = time.perf_counter()
        model.evaluate_safety(scenario['ego_safe'], scenario['agent_gmms'])
        times.append(time.perf_counter() - start)
    results['Full Safety Eval (T=6)'] = np.mean(times)
    
    # 4. With belief update
    model_b = FORCE_OPT(tau=0.95, gamma=0.05, use_belief=True)
    model_b.eta = 2.0
    times = []
    for _ in range(n_trials):
        model_b.c_star_cache.clear()
        model_b.belief_filter.reset()
        start = time.perf_counter()
        model_b.evaluate_safety(scenario['ego_safe'], scenario['agent_gmms'])
        for t_idx in range(scenario['T']):
            model_b.update_belief(scenario['gt_trajectory'][t_idx], scenario['agent_gmms'][t_idx])
        times.append(time.perf_counter() - start)
    results['Safety + Belief (T=6)'] = np.mean(times)
    
    # 5. Worst-case FRS
    times = []
    for _ in range(n_trials):
        start = time.perf_counter()
        compute_worst_case_frs(scenario['agent_start'], scenario['agent_speed'], scenario['dt'], 6)
        times.append(time.perf_counter() - start)
    results['Worst-Case FRS'] = np.mean(times)
    
    print(f"{'Method':<30} {'Mean Time':>12}")
    print("-" * 44)
    for method, t in results.items():
        if t < 1e-3:
            print(f"{method:<30} {t*1e6:>9.1f} μs")
        else:
            print(f"{method:<30} {t*1e3:>9.2f} ms")


benchmark_runtime()

---
## Summary

This notebook implements the full FORCE-OPT algorithm from the paper:

| Component | Section | Key Function / Class |
|---|---|---|
| FRS extraction via convex optimization | IV-B, Thm. 2 | `solve_frs_convex_optimization()` |
| Mahalanobis energy & FRS membership | IV-B | `mahalanobis_energy()`, `is_in_frs()` |
| Non-conformity score | IV-C, Eq. 7 | `compute_nonconformity_score()` |
| Conformal calibration | IV-C, Eq. 8-12 | `calibrate_conformal()` |
| Bayesian confidence filter | IV-D, Eq. 13 | `BayesianConfidenceFilter` |
| Worst-case FRS fallback | IV-D | `compute_worst_case_frs()` |
| Full pipeline | IV | `FORCE_OPT` class |

### To integrate with a real trajectory predictor:
1. Replace `generate_calibration_data()` with real predictor outputs on a held-out calibration split.
2. Feed real GMM outputs from your predictor (e.g., Trajectron++, Autobots) as `GMMDistribution` objects.
3. Call `model.calibrate(...)` once, then `model.evaluate_safety(...)` at test time.